# 03 — Model 2: A0 HingRoBERTa

**Part of the 5-notebook split** of `code-mixed-hate-speech-detection-pipeline-v2.ipynb`.
The complete original notebook is kept unchanged for upload.

**Scope:** Transformer harness (4.1–4.4) + the HingRoBERTa headline fine-tune (4.5), 3 seeds. No other backbone.

**Prerequisites:** 01 (splits); shared metric helpers lifted verbatim from 3.1.

**Writes:** hing-roberta preds/probs/thresholds, model_hing-roberta_seed42.pt, split_state/03_hing_state.json

> Run 01 → 05 in order in the same environment: Kaggle with **Internet ON** and
> **GPU T4 x2**. Every assigned code cell is copied verbatim from the original;
> only the small handoff cells are new. Outputs are intentionally cleared — the
> executed results live in the unchanged original notebook, the PDF, and the
> `results/` artifacts.

### 0.1 Dependencies

In [ ]:
import sys, subprocess

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# transformers>=4.46 required: eval_strategy + processing_class were introduced in 4.46.
pip_install([
    "transformers>=4.46,<5",
    "datasets>=3.0",
    "accelerate>=0.34",
    "scipy>=1.11",
])

### 0.2 Imports & Global Configuration (single source of truth for the environment)

In [ ]:
import os, re, json, math, time, random, hashlib, shutil, warnings, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset as TorchDataset

from datasets import load_dataset
from huggingface_hub import HfApi
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, set_seed as hf_set_seed,
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    f1_score, precision_recall_fscore_support, matthews_corrcoef,
    balanced_accuracy_score, confusion_matrix, classification_report,
)
from scipy import stats

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.dpi"] = 110

# ---- Environment contract: the ONLY place run mode is decided ---------------
SMOKE_TEST = False   # True = tiny data + tiny model, CPU fast pass (~2 min)

PRISM_ID  = "pankajbiswas6/prism-hinglish-hate-speech"
MEHATE_REPO = "l3cube-pune/MarathiNLP"          # GitHub repo, CSV fetched at HEAD SHA
MEHATE_CSV  = "MeEval/L3Cube-MeHate/codemix-hate-small-test.csv"

SEEDS = [42, 43, 44]
TINY_MODEL = "sshleifer/tiny-distilroberta-base"
MODELS = {
    "hing-roberta": "l3cube-pune/hing-roberta",
    "muril":        "google/muril-base-cased",
    "xlm-roberta":  "xlm-roberta-base",
}
if SMOKE_TEST:
    MODELS = {k: TINY_MODEL for k in MODELS}

MAX_LEN     = 128 if not SMOKE_TEST else 32
NUM_EPOCHS  = 3   if not SMOKE_TEST else 1
BATCH_SIZE  = 32  if not SMOKE_TEST else 4
BOOTSTRAP_N = 1_000 if not SMOKE_TEST else 100
SMOKE_N     = 200
LR, WEIGHT_DECAY, WARMUP_RATIO = 2e-5, 0.01, 0.1   # named, not magic

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./kaggle_working")
RESULTS_DIR, FIGURES_DIR, PREDS_DIR = WORK_DIR/"results", WORK_DIR/"figures", WORK_DIR/"preds"
for d in (RESULTS_DIR, FIGURES_DIR, PREDS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Fail fast: the full run is impossible on CPU and would silently drop fp16.
if not SMOKE_TEST:
    assert torch.cuda.is_available(), (
        "GPU required: set Accelerator to 'GPU T4 x2' (or P100) in notebook settings. "
        "For a CPU pass, set SMOKE_TEST = True in section 0.2.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"SMOKE_TEST = {SMOKE_TEST} | Device = {DEVICE} | Work dir = {WORK_DIR.resolve()}")

### 0.3 Reproducibility helpers

In [ ]:
def set_seed(seed: int) -> None:
    """Seed python, numpy and torch (incl. CUDA). Note: with fp16 + cuDNN this is
    reproducible-in-practice, not bit-exact; `full_determinism=True` would cost speed."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

ALL_SEED_RESULTS = []   # one row per (model, seed, split)
PER_CLASS_RESULTS = []  # per-class P/R/F1 rows
COSTS = []              # params / train time / ms-per-item per (model, seed)
THRESH_CURVES = {}      # (model_key, seed) -> (thresholds, val macro-F1 curve)
LOG_HISTORY = {}        # (model_key, seed) -> trainer.state.log_history
TRAINED = {}            # (model_key, seed=SEEDS[0]) -> artifacts needed in sections 6-7

print("Reproducibility helpers ready.")

In [ ]:
# ---- Shared handoff directory for the 5-part notebook chain ------------------
# Run 01 -> 02 -> 03 -> 04 -> 05 in the SAME environment (Kaggle, Internet ON, GPU).
STATE_DIR = WORK_DIR / "split_state"
STATE_DIR.mkdir(parents=True, exist_ok=True)
print("Split state dir:", STATE_DIR)

### Shared helpers (verbatim definitions from the complete notebook)

Only the definitions needed by this part are copied here — no duplicate section
output. Original logic is unchanged.

In [ ]:
# ---- Handoff from 01_data_and_preprocessing.ipynb ---------------------------
# Reconstruct the exact processed `prism` splits the monolith held in memory.
prism = {}
for _k in ("train", "val", "test"):
    _p = RESULTS_DIR / f"prism_{_k}.csv"
    assert _p.exists(), f"Missing {_p} - run 01_data_and_preprocessing.ipynb first"
    _df = pd.read_csv(_p)
    _df["text"] = _df["text"].fillna("").astype(str)
    _df["label"] = _df["label"].astype(int)
    prism[_k] = _df
print("Loaded processed splits:", {k: len(v) for k, v in prism.items()})

In [ ]:
def compute_metrics(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return {"macro_f1": f, "macro_precision": p, "macro_recall": r,
            "mcc": matthews_corrcoef(y_true, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred)}

def per_class_rows(y_true, y_pred, model_label, key, seed, split):
    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    rows = []
    for cls, name in (("0", "non-hate"), ("1", "hate")):
        m = rep[cls]
        rows.append({"model": model_label, "key": key, "seed": seed, "split": split,
                     "class": name, "precision": m["precision"], "recall": m["recall"],
                     "f1": m["f1-score"], "support": m["support"]})
    return rows

def record_run(model_label, key, seed, split, y_true, y_pred, extras=None):
    metrics = compute_metrics(y_true, y_pred)
    ALL_SEED_RESULTS.append({"model": model_label, "key": key, "seed": seed,
                             "split": split, **metrics, **(extras or {})})
    PER_CLASS_RESULTS.extend(per_class_rows(y_true, y_pred, model_label, key, seed, split))
    return metrics

def make_b0(seed: int) -> Pipeline:
    # Char 2-5 gram TF-IDF + LinearSVC. GPU-free, robust to romanisation noise.
    return Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(2, 5), lowercase=True,
                                  min_df=2, max_features=300_000, sublinear_tf=True)),
        ("clf", LinearSVC(C=1.0, random_state=seed)),
    ])

X_train, y_train = prism["train"]["text"].to_numpy(), prism["train"]["label"].to_numpy()

X_val,   y_val   = prism["val"]["text"].to_numpy(),   prism["val"]["label"].to_numpy()

X_test,  y_test  = prism["test"]["text"].to_numpy(),  prism["test"]["label"].to_numpy()

## 4. Advanced A0 — Fine-tuned Transformers

### 4.1 Tokenised torch datasets + transformers metric adapter

In [ ]:
class TextDataset(TorchDataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts, self.labels = list(texts), list(labels)
        self.tok, self.max_len = tokenizer, max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = self.tok(self.texts[i], truncation=True, max_length=self.max_len,
                       padding="max_length", return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[i], dtype=torch.long)
        return item

def compute_metrics_transformers(eval_pred):
    logits, labels = eval_pred
    return compute_metrics(labels, np.argmax(logits, axis=-1))

### 4.2 Training args (extracted — v1 buried this in a 72-line method)

In [ ]:
def build_training_args(model_key: str, seed: int) -> TrainingArguments:
    # Smoke mode: max_steps (4) < one epoch, so eval/save must be step-based for
    # load_best_model_at_end to have a checkpoint to restore.
    if SMOKE_TEST:
        schedule = dict(eval_strategy="steps", eval_steps=2,
                        save_strategy="steps", save_steps=2, max_steps=4)
    else:
        schedule = dict(eval_strategy="epoch", save_strategy="epoch")
    return TrainingArguments(
        output_dir=str(WORK_DIR / f"ckpt_{model_key}_seed{seed}"),
        seed=seed, data_seed=seed,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1", greater_is_better=True,
        learning_rate=LR, weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
        fp16=(DEVICE.type == "cuda" and not SMOKE_TEST),
        logging_steps=50, save_total_limit=1, report_to="none",
        **schedule,
    )

### 4.3 Prediction, threshold tuning (on **validation** only) and checkpointing

In [ ]:
def tune_threshold(y_val_true, val_probs):
    ths = np.arange(0.05, 0.96, 0.01)
    f1s = np.array([f1_score(y_val_true, (val_probs >= t).astype(int),
                             average="macro", zero_division=0) for t in ths])
    return float(ths[int(np.argmax(f1s))]), ths, f1s

def predict_and_checkpoint(trainer, tok, model_key, seed):
    t0 = time.perf_counter()
    val_out  = trainer.predict(TextDataset(X_val,  y_val,  tok, MAX_LEN))
    test_out = trainer.predict(TextDataset(X_test, y_test, tok, MAX_LEN))
    infer_ms = (time.perf_counter() - t0) / len(X_test) * 1000

    val_probs  = torch.softmax(torch.tensor(val_out.predictions),  dim=-1)[:, 1].numpy()
    test_probs = torch.softmax(torch.tensor(test_out.predictions), dim=-1)[:, 1].numpy()

    thr, ths, curve = tune_threshold(y_val, val_probs)      # tuned on VALIDATION only
    val_pred  = (val_probs  >= thr).astype(int)
    test_pred = (test_probs >= thr).astype(int)
    test_argmax = np.argmax(test_out.predictions, axis=-1)  # fixed-0.5 comparison

    np.save(PREDS_DIR / f"{model_key}_seed{seed}_val.npy", val_pred)
    np.save(PREDS_DIR / f"{model_key}_seed{seed}_test.npy", test_pred)
    np.save(PREDS_DIR / f"{model_key}_seed{seed}_test_probs.npy", test_probs)
    np.save(PREDS_DIR / f"{model_key}_seed{seed}_thr.npy", np.array([thr]))
    return val_pred, test_pred, test_probs, test_argmax, thr, ths, curve, infer_ms

### 4.4 One harness, three backbones (memory-safe: only seed-42 model stays resident)

In [ ]:
def run_transformer(model_key: str, model_name: str, seed: int):
    set_seed(seed)
    hf_set_seed(seed)
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6

    trainer = Trainer(model=model, args=build_training_args(model_key, seed),
                      train_dataset=TextDataset(X_train, y_train, tok, MAX_LEN),
                      eval_dataset=TextDataset(X_val, y_val, tok, MAX_LEN),
                      processing_class=tok, compute_metrics=compute_metrics_transformers)
    t0 = time.perf_counter()
    trainer.train()
    train_s = time.perf_counter() - t0
    LOG_HISTORY[(model_key, seed)] = trainer.state.log_history

    val_pred, test_pred, test_probs, test_argmax, thr, ths, curve, infer_ms = \
        predict_and_checkpoint(trainer, tok, model_key, seed)
    THRESH_CURVES[(model_key, seed)] = (ths, curve)

    extras = {"threshold": thr, "test_macro_f1_argmax":
              compute_metrics(y_test, test_argmax)["macro_f1"]}
    val_m  = record_run(model_key, model_key, seed, "val",  y_val,  val_pred,  extras)
    test_m = record_run(model_key, model_key, seed, "test", y_test, test_pred, extras)
    COSTS.append({"model": model_key, "key": model_key, "seed": seed,
                  "params_m": n_params, "train_seconds": train_s, "ms_per_item": infer_ms})

    if seed == SEEDS[0]:
        # Only the seed-42 artefacts are reused later (sections 6-7): keep resident.
        TRAINED[(model_key, seed)] = {"tokenizer": tok, "model": trainer.model,
                                      "val_pred": val_pred, "test_pred": test_pred,
                                      "test_probs": test_probs, "threshold": thr}
    else:
        del trainer, model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    shutil.rmtree(WORK_DIR / f"ckpt_{model_key}_seed{seed}", ignore_errors=True)
    print(f"[{model_key} seed {seed}]  val macro-F1 {val_m['macro_f1']:.4f} | "
          f"test macro-F1 {test_m['macro_f1']:.4f} @ thr={thr:.2f} | "
          f"train {train_s:.0f}s | {infer_ms:.2f} ms/item")

### 4.5 A0 headline — HingRoBERTa (pretrained on ~1B tokens of romanised Hinglish)

In [ ]:
for seed in SEEDS:
    run_transformer("hing-roberta", MODELS["hing-roberta"], seed)

In [ ]:
# ---- Split handoff (03_hing -> 05) ----------------------------------------------
def _jsonable(x):
    # numpy scalars -> plain python so json.dumps keeps numbers numeric
    if isinstance(x, dict):
        return {str(k): _jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_jsonable(v) for v in x]
    if isinstance(x, np.generic):
        return x.item()
    return x

_curves = {f"{k[0]}|{k[1]}": {"ths": v[0].tolist(), "curve": v[1].tolist()} for k, v in THRESH_CURVES.items()}
_logs = {f"{k[0]}|{k[1]}": v for k, v in LOG_HISTORY.items()}
(STATE_DIR / "03_hing_state.json").write_text(json.dumps(_jsonable({
    "all_seed": ALL_SEED_RESULTS,
    "per_class": PER_CLASS_RESULTS,
    "costs": COSTS,
    "thresh_curves": _curves,
    "log_history": _logs,
}), indent=2))
print("Saved 03_hing_state.json:", len(ALL_SEED_RESULTS), "result rows,", len(COSTS), "cost rows")
# Persist the seed-42 trained model so notebook 05 can reproduce section 7.2 exactly.
for (_mk, _sd), _art in TRAINED.items():
    torch.save(_art["model"].state_dict(), STATE_DIR / f"model_{_mk}_seed{_sd}.pt")
    print("Saved model state:", f"model_{_mk}_seed{_sd}.pt")